In [1]:
import os
from typing_extensions import TypedDict
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langgraph.graph import StateGraph, START, END
from langchain_community.document_loaders import UnstructuredMarkdownLoader
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_community.llms import Ollama
from langchain_ollama import ChatOllama
from langchain_community.graphs import Neo4jGraph

c:\Users\mahes\StudioProjects\graph_db_trial\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\mahes\AppData\Local\Temp\ipykernel_15908\2659311486.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import UnstructuredMarkdownLoader
C:\Users\mahes\AppData\Local\Temp\ipykernel_15908\2659311486.py:6: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.graph_transformers import LLMGraphTransformer


# 1. Define the workflow state

In [2]:
class GraphState(TypedDict):
    file_path: str
    raw_text: str
    chunks: list             # Added to hold document chunks
    extracted_graph_docs: list


# 2. Load mardown data

In [3]:
def load_markdown_node(state: GraphState) -> dict:
    """Reads the educational markdown chapter using native file tools."""
    # This completely eliminates the reliance on unstructured/defusedxml
    with open(state["file_path"], "r", encoding="utf-8") as f:
        combined_text = f.read()
    return {"raw_text": combined_text}


# 3. Split raw data to multiple chunks

In [4]:
def chunk_markdown_node(state: GraphState) -> dict:
    """Splits raw text into structured markdown chunks."""
    # FIXED: Grab from raw_text instead of non-existent chunks key
    raw_text = state["raw_text"] 
    
    headers_to_split_on = [
        ("#", "Header_1"),
        ("##", "Header_2"),
        ("###", "Header_3"),
    ]
    
    markdown_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=headers_to_split_on, 
        strip_headers=False
    )
    
    document_chunks = markdown_splitter.split_text(raw_text)
    return {"chunks": document_chunks}


# 4. Nodes and relationship extraction

In [5]:
model = "qwen3.5:2b"

### 4.1 Basic extraction process

In [6]:
def extract_entities_node_basic(state: GraphState) -> dict:
    """Extracts nodes and relationships tailored for educational domains."""
    
    # 2. Use ChatOllama with a model supporting structured formatting
    # Adjust "qwen2.5" or "llama3.1" depending on what you have pulled locally.
    # We set temperature=0 for precise structural parsing.
    llm = ChatOllama(
        model=model, 
        temperature=0,
        format="json"  # Forces the local backend into JSON compilation mode
    )
    
    # 3. Create the transformer. We add 'ignore_tool_usage=True' to tell 
    # LangChain to fall back to structured JSON extraction prompts rather than 
    # OpenAI-specific function schemas, which local models prefer.
    transformer = LLMGraphTransformer(
        llm=llm,
        allowed_nodes=["Topic", "Subtopic", "Concept", "Definition", "Formula", "Theorem", "Example"],
        allowed_relationships=["COVERS", "EXPLAINS", "PREREQUISITE_FOR", "PART_OF", "USES_FORMULA", "ILLUSTRATES"],
        ignore_tool_usage=True  # <-- CRITICAL for local Ollama models
    )

    # Process the text chunks
    graph_documents = transformer.convert_to_graph_documents(state["chunks"])
    
    # Quick sanity debug check in your terminal
    print(f"\n--- Extracted {len(graph_documents)} Graph Docs ---")
    for gd in graph_documents:
        print(f"Nodes found: {len(gd.nodes)} | Edges found: {len(gd.relationships)}")
        
    return {"extracted_graph_docs": graph_documents}

### 4.2 Extraction with custom transformer

In [49]:
from pydantic import BaseModel, Field
from typing import List

# 1. Outline the explicit graph target structure for Pydantic
class KnowledgeNode(BaseModel):
    id: str = Field(description="The unique name or key identifier of the entity")
    name: str = Field(description="The human-readable name of the entity")
    type: str = Field(description="Must be one of: Topic, Subtopic, Concept, Definition, Formula, Theorem, Example")

class KnowledgeRelationship(BaseModel):
    source: str = Field(description="The id of the source node")
    target: str = Field(description="The id of the target node")
    type: str = Field(description="Must be one of: COVERS, EXPLAINS, PREREQUISITE_FOR, PART_OF, USES_FORMULA, ILLUSTRATES")

class EducationalGraph(BaseModel):
    nodes: List[KnowledgeNode] = Field(default_factory=list)
    relationships: List[KnowledgeRelationship] = Field(default_factory=list)

def extract_entities_node_custom_transformer(state: GraphState) -> dict:
    from langchain_community.graphs.graph_document import GraphDocument, Node, Relationship
    
    # Structured bindings handle markdown codeblocks from Ollama correctly
    llm = ChatOllama(model=model, temperature=0)
    structured_llm = llm.with_structured_output(EducationalGraph, method="json_schema")
    
    all_extracted_docs = []
    
    for chunk in state["chunks"]:
        # prompt = f"""You are a top-tier algorithm designed for extracting information in
        # structured formats to build a knowledge graph. Your task is to identify 
        # the entities and relations requested with the user prompt from a given 
        # text.
        # Attempt to extract as many entities and relations as you can. Maintain 
        # Entity Consistency: When extracting entities, it's vital to ensure 
        # consistency.
        
        # The knowledge graph should be coherent and easily understandable, 
        # so maintaining consistency in entity references is crucial.
        # IMPORTANT NOTES:\n- Don't add any explanation and text.
        
        prompt = f"""Extract all relevant educational nodes and relationships from this text.

        Allowed Nodes: Topic, Subtopic, Concept, Definition, Formula, Theorem, Example
        Allowed Relationships: COVERS, EXPLAINS, PREREQUISITE_FOR, PART_OF, USES_FORMULA, ILLUSTRATES
        
        Text content:
        {chunk.page_content}"""
        
        try:
            # Executes schema extraction safely
            result = structured_llm.invoke(prompt)
            # print(f"Extracted {len(result.nodes)} nodes and {len(result.relationships)} relationships from chunk.")
            # print(f"Nodes: {[n for n in result.nodes]}")
            # print(f"Relationships: {[r for r in result.relationships]}")
            
            # Map back to standard LangChain format for your Neo4j node step
            langchain_nodes = [Node(id=n.id, type=n.type, properties={'name': n.name}) for n in result.nodes]
            nodeDict = {n.id: n for n in langchain_nodes}  # For quick lookup
            langchain_rels = [
                Relationship(
                    source=nodeDict[r.source], 
                    target=nodeDict[r.target], 
                    type=r.type
                ) for r in result.relationships
            ]
            
            graph_doc = GraphDocument(nodes=langchain_nodes, relationships=langchain_rels, source=chunk)
            all_extracted_docs.append(graph_doc)
        except Exception as e:
            print(f"Skipping a chunk due to parsing error: {e}")
            
    print(f"\n--- Native Extracted {len(all_extracted_docs)} Docs ---")
    return {"extracted_graph_docs": all_extracted_docs}

# 5. Save data to DB

In [7]:
def save_to_neo4j_node(state: GraphState) -> dict:
    """Saves the extracted graph data into Neo4j."""
    graph = Neo4jGraph(username="neo4j", password="12345678", url="neo4j://localhost:7687", database="langchain-ontology-qwen")
    graph.add_graph_documents(state["extracted_graph_docs"])
    return {}

# 6. Pipeline configuration

### 6.1 With Basic extractor

In [ ]:
workflow = StateGraph(GraphState)

# Add all nodes (FIXED: Added the chunking node)
workflow.add_node("load_markdown", load_markdown_node)
workflow.add_node("chunk_markdown", chunk_markdown_node)
workflow.add_node("extract_entities", extract_entities_node_basic)  # or extract_entities_node_custom_transformer
workflow.add_node("save_to_neo4j", save_to_neo4j_node)

# Route the linear workflow graph correctly
workflow.add_edge(START, "load_markdown")
workflow.add_edge("load_markdown", "chunk_markdown")
workflow.add_edge("chunk_markdown", "extract_entities")
workflow.add_edge("extract_entities", "save_to_neo4j")
workflow.add_edge("save_to_neo4j", END)

### 6.2 With Custom Extractor

In [ ]:
workflow = StateGraph(GraphState)

# Add all nodes (FIXED: Added the chunking node)
workflow.add_node("load_markdown", load_markdown_node)
workflow.add_node("chunk_markdown", chunk_markdown_node)
workflow.add_node("extract_entities", extract_entities_node_custom_transformer)
workflow.add_node("save_to_neo4j", save_to_neo4j_node)

# Route the linear workflow graph correctly
workflow.add_edge(START, "load_markdown")
workflow.add_edge("load_markdown", "chunk_markdown")
workflow.add_edge("chunk_markdown", "extract_entities")
workflow.add_edge("extract_entities", "save_to_neo4j")
workflow.add_edge("save_to_neo4j", END)

# 7. Runner

In [ ]:
# Compile the graph
chapter_ingestion_app = workflow.compile()

# Example Usage:
config = {"file_path": "processed_notes_with_descriptions.md"}
chapter_ingestion_app.invoke(config)

# 8. Debug runner

In [20]:
config = {"file_path": "processed_notes_with_descriptions.md"}

config = load_markdown_node(config)
config = chunk_markdown_node(config)

config['chunks'] = [config['chunks'][6]]

config

{'chunks': [Document(metadata={'Header_1': '7.5 ACCELERATION DUE TO GRAVITY OF THE EARTH', 'Header_3': '7.6 ACCELERATION DUE TO GRAVITY BELOW AND ABOVE THE SURFACE OF EARTH'}, page_content='### 7.6 ACCELERATION DUE TO GRAVITY BELOW AND ABOVE THE SURFACE OF EARTH  \nConsider a point mass *m* at a height *h* above the surface of the earth as shown in Fig. 7.8(a). The radius of the earth is denoted by *RE .* Since this point is outside the earth,  \n*Fig. 7.8 (a) g at a height h above the surface of the earth.*  \nits distance from the centre of the earth is (*RE + h* ). If *F* (*h*) denoted the magnitude of the force on the point mass *m* , we get from Eq. (7.5) :  \n$$F(h) = \\frac{GM_E m}{(R_E + h)^2} \\quad (7.13)$$  \nThe acceleration experienced by the point mass is *Fh m gh* ( )/ ( ) ≡ and we get  \n$$g(h) = \\frac{F(h)}{m} = \\frac{GM_E}{(R_E + h)^2}. \\quad (7.14)$$  \nThis is clearly less than the value of g on the surface of earth : 2 . *<sup>E</sup> E GM <sup>g</sup> <sup>R</s

### 8.1 With System context 16K

In [50]:
import time

print("\nStarting extraction for model: rnj-1")
model = 'rnj-1'
start_time = time.time()
basic_rnj_1 = extract_entities_node_custom_transformer(config)
end_time = time.time()
print (f"Extracted {len(basic_rnj_1['extracted_graph_docs'][0].nodes)} graph documents using custom transformer.")
for n in basic_rnj_1['extracted_graph_docs'][0].nodes:
    print(n)
print (f"Extracted {len(basic_rnj_1['extracted_graph_docs'][0].relationships)} relationships using custom transformer.")
for r in basic_rnj_1['extracted_graph_docs'][0].relationships:
    print(r)
print (f"Execution time: {end_time - start_time} seconds.")

print("\nStarting extraction for model: qwen3.5:2b")
model = 'qwen3.5:2b'
start_time = time.time()
qwen_2b = extract_entities_node_custom_transformer(config)
end_time = time.time()
print (f"Extracted {len(qwen_2b['extracted_graph_docs'][0].nodes)} graph documents using custom transformer.")
for n in qwen_2b['extracted_graph_docs'][0].nodes:
    print(n)
print (f"Extracted {len(qwen_2b['extracted_graph_docs'][0].relationships)} relationships using custom transformer.")
for r in qwen_2b['extracted_graph_docs'][0].relationships:
    print(r)
print (f"Execution time: {end_time - start_time} seconds.")

print("\nStarting extraction for model: qwen3.5:4b")
model = 'qwen3.5:4b'
start_time = time.time()
qwen_4b = extract_entities_node_custom_transformer(config)
end_time = time.time()
print (f"Extracted {len(qwen_4b['extracted_graph_docs'][0].nodes)} graph documents using custom transformer.")
for n in qwen_4b['extracted_graph_docs'][0].nodes:
    print(n)
print (f"Extracted {len(qwen_4b['extracted_graph_docs'][0].relationships)} relationships using custom transformer.")
for r in qwen_4b['extracted_graph_docs'][0].relationships:
    print(r)
print (f"Execution time: {end_time - start_time} seconds.")

print("\nStarting extraction for model: qwen3.5")
model = 'qwen3.5'
start_time = time.time()
qwen_8b = extract_entities_node_custom_transformer(config)
end_time = time.time()
print (f"Extracted {len(qwen_8b['extracted_graph_docs'][0].nodes)} graph documents using custom transformer.")
for n in qwen_8b['extracted_graph_docs'][0].nodes:
    print(n)
print (f"Extracted {len(qwen_8b['extracted_graph_docs'][0].relationships)} relationships using custom transformer.")
for r in qwen_8b['extracted_graph_docs'][0].relationships:
    print(r)
print (f"Execution time: {end_time - start_time} seconds.")



Starting extraction for model: rnj-1

--- Native Extracted 1 Docs ---
Extracted 4 graph documents using custom transformer.
id='Topic: Gravitational Potential Energy' type='Topic' properties={'name': 'Gravitational Potential Energy'}
id='Subtopic: Definition and Concept' type='Subtopic' properties={'name': 'Definition and Concept'}
id='Concept: Gravitational Potential Energy as Work Done' type='Concept' properties={'name': 'Gravitational Potential Energy as Work Done'}
id='Formula: W = -Gm1m2/r' type='Formula' properties={'name': 'W = -Gm1m2/r'}
Extracted 3 relationships using custom transformer.
source=Node(id='Topic: Gravitational Potential Energy', type='Topic', properties={'name': 'Gravitational Potential Energy'}) target=Node(id='Subtopic: Definition and Concept', type='Subtopic', properties={'name': 'Definition and Concept'}) type='IS_A' properties={}
source=Node(id='Subtopic: Definition and Concept', type='Subtopic', properties={'name': 'Definition and Concept'}) target=Node(id

### 8.2 With System context 8K

In [51]:
import time

print("\nStarting extraction for model: rnj-1")
model = 'rnj-1'
start_time = time.time()
basic_rnj_1 = extract_entities_node_custom_transformer(config)
end_time = time.time()
print (f"Extracted {len(basic_rnj_1['extracted_graph_docs'][0].nodes)} graph documents using custom transformer.")
for n in basic_rnj_1['extracted_graph_docs'][0].nodes:
    print(n)
print (f"Extracted {len(basic_rnj_1['extracted_graph_docs'][0].relationships)} relationships using custom transformer.")
for r in basic_rnj_1['extracted_graph_docs'][0].relationships:
    print(r)
print (f"Execution time: {end_time - start_time} seconds.")

print("\nStarting extraction for model: qwen3.5:2b")
model = 'qwen3.5:2b'
start_time = time.time()
qwen_2b = extract_entities_node_custom_transformer(config)
end_time = time.time()
print (f"Extracted {len(qwen_2b['extracted_graph_docs'][0].nodes)} graph documents using custom transformer.")
for n in qwen_2b['extracted_graph_docs'][0].nodes:
    print(n)
print (f"Extracted {len(qwen_2b['extracted_graph_docs'][0].relationships)} relationships using custom transformer.")
for r in qwen_2b['extracted_graph_docs'][0].relationships:
    print(r)
print (f"Execution time: {end_time - start_time} seconds.")

print("\nStarting extraction for model: qwen3.5:4b")
model = 'qwen3.5:4b'
start_time = time.time()
qwen_4b = extract_entities_node_custom_transformer(config)
end_time = time.time()
print (f"Extracted {len(qwen_4b['extracted_graph_docs'][0].nodes)} graph documents using custom transformer.")
for n in qwen_4b['extracted_graph_docs'][0].nodes:
    print(n)
print (f"Extracted {len(qwen_4b['extracted_graph_docs'][0].relationships)} relationships using custom transformer.")
for r in qwen_4b['extracted_graph_docs'][0].relationships:
    print(r)
print (f"Execution time: {end_time - start_time} seconds.")

print("\nStarting extraction for model: qwen3.5")
model = 'qwen3.5'
start_time = time.time()
qwen_8b = extract_entities_node_custom_transformer(config)
end_time = time.time()
print (f"Extracted {len(qwen_8b['extracted_graph_docs'][0].nodes)} graph documents using custom transformer.")
for n in qwen_8b['extracted_graph_docs'][0].nodes:
    print(n)
print (f"Extracted {len(qwen_8b['extracted_graph_docs'][0].relationships)} relationships using custom transformer.")
for r in qwen_8b['extracted_graph_docs'][0].relationships:
    print(r)
print (f"Execution time: {end_time - start_time} seconds.")



Starting extraction for model: rnj-1

--- Native Extracted 1 Docs ---
Extracted 5 graph documents using custom transformer.
id='Topic: Gravitational Potential Energy' type='Topic' properties={'name': 'Gravitational Potential Energy'}
id='Subtopic: Definition and Concept' type='Subtopic' properties={'name': 'Definition and Concept'}
id='Concept: Gravitational Potential Energy as Work Done' type='Concept' properties={'name': 'Gravitational Potential Energy as Work Done'}
id='Formula: Gravitational Potential Energy' type='Formula' properties={'name': 'Gravitational Potential Energy Formula'}
id='Example: Calculation of Gravitational Potential Energy' type='Example' properties={'name': 'Calculation Example'}
Extracted 4 relationships using custom transformer.
source=Node(id='Topic: Gravitational Potential Energy', type='Topic', properties={'name': 'Gravitational Potential Energy'}) target=Node(id='Subtopic: Definition and Concept', type='Subtopic', properties={'name': 'Definition and Conc

IndexError: list index out of range

### 8.3 With System context 4K

In [52]:
import time

print("\nStarting extraction for model: rnj-1")
model = 'rnj-1'
start_time = time.time()
basic_rnj_1 = extract_entities_node_custom_transformer(config)
end_time = time.time()
print (f"Extracted {len(basic_rnj_1['extracted_graph_docs'][0].nodes)} graph documents using custom transformer.")
for n in basic_rnj_1['extracted_graph_docs'][0].nodes:
    print(n)
print (f"Extracted {len(basic_rnj_1['extracted_graph_docs'][0].relationships)} relationships using custom transformer.")
for r in basic_rnj_1['extracted_graph_docs'][0].relationships:
    print(r)
print (f"Execution time: {end_time - start_time} seconds.")

print("\nStarting extraction for model: qwen3.5:2b")
model = 'qwen3.5:2b'
start_time = time.time()
qwen_2b = extract_entities_node_custom_transformer(config)
end_time = time.time()
print (f"Extracted {len(qwen_2b['extracted_graph_docs'][0].nodes)} graph documents using custom transformer.")
for n in qwen_2b['extracted_graph_docs'][0].nodes:
    print(n)
print (f"Extracted {len(qwen_2b['extracted_graph_docs'][0].relationships)} relationships using custom transformer.")
for r in qwen_2b['extracted_graph_docs'][0].relationships:
    print(r)
print (f"Execution time: {end_time - start_time} seconds.")

print("\nStarting extraction for model: qwen3.5:4b")
model = 'qwen3.5:4b'
start_time = time.time()
qwen_4b = extract_entities_node_custom_transformer(config)
end_time = time.time()
print (f"Extracted {len(qwen_4b['extracted_graph_docs'][0].nodes)} graph documents using custom transformer.")
for n in qwen_4b['extracted_graph_docs'][0].nodes:
    print(n)
print (f"Extracted {len(qwen_4b['extracted_graph_docs'][0].relationships)} relationships using custom transformer.")
for r in qwen_4b['extracted_graph_docs'][0].relationships:
    print(r)
print (f"Execution time: {end_time - start_time} seconds.")

print("\nStarting extraction for model: qwen3.5")
model = 'qwen3.5'
start_time = time.time()
qwen_8b = extract_entities_node_custom_transformer(config)
end_time = time.time()
print (f"Extracted {len(qwen_8b['extracted_graph_docs'][0].nodes)} graph documents using custom transformer.")
for n in qwen_8b['extracted_graph_docs'][0].nodes:
    print(n)
print (f"Extracted {len(qwen_8b['extracted_graph_docs'][0].relationships)} relationships using custom transformer.")
for r in qwen_8b['extracted_graph_docs'][0].relationships:
    print(r)
print (f"Execution time: {end_time - start_time} seconds.")



Starting extraction for model: rnj-1

--- Native Extracted 1 Docs ---
Extracted 6 graph documents using custom transformer.
id='Topic: Gravitational Potential Energy' type='Topic' properties={'name': 'Gravitational Potential Energy'}
id='Subtopic: Definition and Concept' type='Subtopic' properties={'name': 'Definition and Concept'}
id='Concept: Gravitational Potential Energy as Work Done' type='Concept' properties={'name': 'Gravitational Potential Energy as Work Done'}
id='Concept: Calculation of Potential Energy' type='Concept' properties={'name': 'Calculation of Potential Energy'}
id='Subtopic: Superposition Principle' type='Subtopic' properties={'name': 'Superposition Principle'}
id='Example: System of Particles' type='Example' properties={'name': 'System of Particles'}
Extracted 4 relationships using custom transformer.
source=Node(id='Topic: Gravitational Potential Energy', type='Topic', properties={'name': 'Gravitational Potential Energy'}) target=Node(id='Subtopic: Definition a

IndexError: list index out of range

Results With respect to time

| Model | status | Accuracy | Time Taken | No of Nodes | No of Relationships |
| :--- | :---: | ---: | ---: | ---: | ---: |
|qwen3.5:8b 256k (system - 16k)| success |  | 206 | 5 | 2 |
|qwen3.5:8b 256k (system - 8k)| failed |  |  |  |  |
|qwen3.5:8b 256k (system - 4k)| failed |  |  |  |  |
|qwen3.5:4b 256k (system - 16k)| success |  | 93 | 16 | 10 |
|qwen3.5:4b 256k (system - 8k)| success |  | 102 | 16 | 5 |
|qwen3.5:4b 256k (system - 4k)| failed |  |  |  |  |
|qwen3.5:2b 256k (system - 16k)| success |  | 75 | 6 | 5 |
|qwen3.5:2b 256k (system - 8k)| success |  | 68 | 1 | 0 |
|qwen3.5:2b 256k (system - 4k)| failed |  |  |  |  |
|rnj-1 8B 32k (system - 16k)| success |  | 29  | 4 | 3 |
|rnj-1 8B 32k (system - 8k)| success |  | 18 | 5 | 4 |
|rnj-1 8B 32k (system - 4k)| success |  | 21 | 6 | 4 |